# Evaluate Multi-Molecule SFT On LPM24

This notebook evaluates the LPM24 multi-SFT checkpoint directly, without PPO or GFlowNet post-training. It reuses the `gflownet_v2.4` Colab bootstrap flow to install dependencies, prepare LPM24 files, and download the same upstream multi-SFT checkpoint used by `06_v2`, then computes validation-glance and sampled processed-test metrics plus parsed stage-count and trajectory-length proxies.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet_v2.4"
REPO_DIR = Path("/content/Thesis")

%cd /content
if (REPO_DIR / ".git").exists():
    print(f"Reusing {REPO_DIR}")
elif REPO_DIR.exists():
    raise RuntimeError(f"Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", REPO_URL, REPO_BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, "FETCH_HEAD"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", REPO_URL, REPO_BRANCH], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print({"repo_dir": str(REPO_DIR), "repo_branch": REPO_BRANCH})


In [ ]:
from datetime import datetime

DATASET_MODE = "never"
CONFIG_OVERRIDE = Path("configs/multi_molecule_gflownet_lpm24.yaml")
SFT_CONFIG = Path("configs/multi_molecule_sft_lpm24.yaml")
DEFAULT_CONFIG_STEM = "multi_molecule_sft_lpm24_eval"
COLAB_OUTPUT_ROOT = Path("/content/drive/MyDrive/multi_molecule_sft_lpm24_eval")
RUN_STAMP = datetime.now().strftime("%y%m%d_%H%M%S")
RUN_NAME = f"{DEFAULT_CONFIG_STEM}_{RUN_STAMP}"
COLAB_OUTPUT_DIR = COLAB_OUTPUT_ROOT / RUN_NAME

GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE = "1jCIVYbzgTw7xQAWvv6SfwM8Y1vL47PDg"
VALIDATION_FRACTION = 0.05
SPLIT_SEED = 42
MAX_TARGET_SYMBOLS = 1024
MAX_STAGE_SYMBOLS = 135

EVAL_SEED = 42
VALIDATION_NUM_EXAMPLES = 128
PROCESSED_TEST_FRACTION = 0.70
PROCESSED_TEST_MAX_EXAMPLES = None
PROCESSED_TEST_SAMPLE_WITH_REPLACEMENT = False
EVAL_REPORT_EVERY_EXAMPLES = 50
# Prompt-level batching for evaluation generation.
EVAL_BATCH_SIZE = 2
GENERATION_NUM_BEAMS = 4
GENERATION_NUM_RETURN_SEQUENCES = 4
GENERATION_MAX_NEW_TOKENS = 1024
GENERATION_EARLY_STOPPING = True
ACCEPTANCE_DICE_THRESHOLD = 0.7
N_CIRCLES_TANIMOTO_THRESHOLD = 0.6

WANDB_PROJECT_URL = "https://wandb.ai/koala-team/Thesis-eval"
WANDB_PROJECT = "Thesis-eval"
WANDB_ENTITY = "koala-team"

OUTPUT_DIR = COLAB_OUTPUT_DIR.expanduser()
OUTPUT_DIR_ZIP = OUTPUT_DIR / f"{OUTPUT_DIR.name}.zip"
DIAGNOSTICS_DIR = OUTPUT_DIR / "diagnostics"
VALIDATION_METRICS_PATH = DIAGNOSTICS_DIR / "sft_validation_evaluation_metrics.json"
VALIDATION_GENERATIONS_PATH = DIAGNOSTICS_DIR / "sft_validation_generations.jsonl"
VALIDATION_PROGRESS_METRICS_PATH = DIAGNOSTICS_DIR / "sft_validation_progress_metrics.jsonl"
PROCESSED_TEST_METRICS_PATH = DIAGNOSTICS_DIR / "sft_processed_test_evaluation_metrics.json"
PROCESSED_TEST_GENERATIONS_PATH = DIAGNOSTICS_DIR / "sft_processed_test_generations.jsonl"
PROCESSED_TEST_PROGRESS_METRICS_PATH = DIAGNOSTICS_DIR / "sft_processed_test_progress_metrics.jsonl"

DEFAULT_SFT_FALLBACK_CHECKPOINT = "QizhiPei/biot5-plus-base"
UPSTREAM_CHECKPOINT = REPO_DIR / "outputs" / "multi_molecule_sft_lpm24" / "checkpoints" / "best"
LPM24_DATASET_DIR = REPO_DIR / "data" / "lpm24"
GROUPED_SPLITS_DIR = LPM24_DATASET_DIR / "grouped_splits"
VALIDATION_DATASET_PATH = GROUPED_SPLITS_DIR / "validation_multimol.jsonl"
PROCESSED_TEST_DATASET_PATH = LPM24_DATASET_DIR / "processed" / "test_multimol.jsonl"
PROCESSED_DATASET_CHECKS = {
    "train_multimol": LPM24_DATASET_DIR / "processed" / "train_multimol.jsonl",
    "test_multimol": LPM24_DATASET_DIR / "processed" / "test_multimol.jsonl",
    "test_eval_first_1000": LPM24_DATASET_DIR / "processed" / "test_eval_first_1000_multimol.jsonl",
}
GROUPED_SPLIT_CHECKS = {
    "train_multimol": GROUPED_SPLITS_DIR / "train_multimol.jsonl",
    "validation_multimol": GROUPED_SPLITS_DIR / "validation_multimol.jsonl",
    "test_multimol": GROUPED_SPLITS_DIR / "test_multimol.jsonl",
}

print({
    "sft_config": str(SFT_CONFIG),
    "bootstrap_config": str(CONFIG_OVERRIDE),
    "run_name": RUN_NAME,
    "output_dir": str(OUTPUT_DIR),
    "upstream_checkpoint": str(UPSTREAM_CHECKPOINT),
    "checkpoint_download_source_configured": bool(GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip()),
    "validation_dataset_path": str(VALIDATION_DATASET_PATH),
    "processed_test_dataset_path": str(PROCESSED_TEST_DATASET_PATH),
    "validation_num_examples": VALIDATION_NUM_EXAMPLES,
    "processed_test_fraction": PROCESSED_TEST_FRACTION,
    "eval_report_every_examples": EVAL_REPORT_EVERY_EXAMPLES,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "generation_num_beams": GENERATION_NUM_BEAMS,
    "generation_num_return_sequences": GENERATION_NUM_RETURN_SEQUENCES,
    "generation_max_new_tokens": GENERATION_MAX_NEW_TOKENS,
    "wandb_project_url": WANDB_PROJECT_URL,
})


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DIAGNOSTICS_DIR, exist_ok=True)

print({
    "colab_output_root": str(COLAB_OUTPUT_ROOT),
    "run_name": RUN_NAME,
    "colab_output_dir": str(COLAB_OUTPUT_DIR),
    "output_dir_exists": OUTPUT_DIR.exists(),
    "diagnostics_dir_exists": DIAGNOSTICS_DIR.exists(),
})


In [ ]:
import os
import wandb

WANDB_API_KEY = "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"
if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY, relogin=True)

print({
    "wandb_api_key_configured": bool(WANDB_API_KEY),
    "wandb_project_url": WANDB_PROJECT_URL,
})


In [ ]:
%cd {REPO_DIR}
bootstrap_command = [
    sys.executable,
    "scripts/init_colab.py",
    "--stage",
    "gflownet",
    "--repo-url",
    REPO_URL,
    "--repo-branch",
    REPO_BRANCH,
    "--repo-dir",
    str(REPO_DIR),
    "--dataset-mode",
    DATASET_MODE,
    "--config",
    str(CONFIG_OVERRIDE),
]
if GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip():
    bootstrap_command.extend([
        "--gflownet-checkpoint-download-source",
        GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE.strip(),
    ])

print("Bootstrapping:", " ".join(str(part) for part in bootstrap_command))

def run_and_stream(command):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Failed to capture command output.")

    try:
        for line in process.stdout:
            print(line, end="", flush=True)
    finally:
        process.stdout.close()

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

run_and_stream(bootstrap_command)

processed_dataset_ready = all(path.exists() for path in PROCESSED_DATASET_CHECKS.values())
if not processed_dataset_ready:
    download_command = [
        sys.executable,
        "scripts/download_lpm24.py",
        "--output-dir",
        str(LPM24_DATASET_DIR),
    ]
    print("Preparing processed LPM24 dataset:", " ".join(str(part) for part in download_command))
    run_and_stream(download_command)
else:
    print("Reusing processed LPM24 dataset:", str(LPM24_DATASET_DIR))

export_command = [
    sys.executable,
    "scripts/prepare_lpm24_training_splits.py",
    "--input-dir",
    str(LPM24_DATASET_DIR),
    "--validation-fraction",
    str(VALIDATION_FRACTION),
    "--seed",
    str(SPLIT_SEED),
    "--max-target-symbols",
    str(MAX_TARGET_SYMBOLS),
    "--max-stage-symbols",
    str(MAX_STAGE_SYMBOLS),
]
print("Preparing grouped validation split:", " ".join(str(part) for part in export_command))
run_and_stream(export_command)

if not UPSTREAM_CHECKPOINT.exists():
    raise RuntimeError(
        "Expected the multi-SFT checkpoint downloaded for 06_v2 at "
        f"{UPSTREAM_CHECKPOINT}. Check GFLOWNET_CHECKPOINT_DOWNLOAD_SOURCE."
    )

print({
    "upstream_checkpoint": str(UPSTREAM_CHECKPOINT),
    "upstream_checkpoint_exists": UPSTREAM_CHECKPOINT.exists(),
    "processed_dataset_checks": {name: path.exists() for name, path in PROCESSED_DATASET_CHECKS.items()},
    "grouped_split_checks": {name: path.exists() for name, path in GROUPED_SPLIT_CHECKS.items()},
})


In [ ]:
import json
import random
from pathlib import Path

import torch
import wandb
from tqdm.auto import tqdm
from transformers import AutoTokenizer, T5ForConditionalGeneration

from evaluation_metrics import (
    EvaluationMetricConfig,
    GenerationGroup,
    MoleculeInput,
    evaluate_generation_groups,
)
from post_training.sft_multi.dataset import MultiMoleculeDataset
from post_training.shared.sequence import parse_staged_target
from src.io_utils import write_jsonl
from src.training import choose_device


def _clean_generated_sequence_text(text: str, tokenizer) -> str:
    cleaned = str(text)
    for token in (tokenizer.pad_token, tokenizer.eos_token, tokenizer.bos_token, "<pad>", "</s>", "<s>"):
        if token:
            cleaned = cleaned.replace(str(token), "")
    return cleaned.strip()


def _select_examples(
    dataset: MultiMoleculeDataset,
    *,
    max_examples: int | None,
    fraction: float | None,
    seed: int,
    with_replacement: bool,
) -> tuple[list[dict[str, object]], list[int]]:
    examples = [dataset[index] for index in range(len(dataset))]
    dataset_size = len(examples)
    if dataset_size == 0:
        return [], []

    if fraction is not None:
        selected_count = int(round(dataset_size * float(fraction)))
    elif max_examples is not None:
        selected_count = int(max_examples)
    else:
        selected_count = dataset_size

    selected_count = max(1, min(selected_count, dataset_size))
    rng = random.Random(int(seed))

    if selected_count >= dataset_size:
        selected_indices = list(range(dataset_size))
    elif with_replacement:
        selected_indices = [rng.randrange(dataset_size) for _ in range(selected_count)]
    else:
        selected_indices = rng.sample(range(dataset_size), selected_count)

    return [examples[index] for index in selected_indices], selected_indices


def _batched(items: list[dict[str, object]], batch_size: int):
    for start in range(0, len(items), int(batch_size)):
        yield items[start : start + int(batch_size)]


def _generate_text_batch(model, tokenizer, batch: list[dict[str, object]], *, device: torch.device) -> list[list[str]]:
    prompts = [str(example["prompt"]) for example in batch]
    inputs = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=int(GENERATION_MAX_NEW_TOKENS),
            num_beams=int(GENERATION_NUM_BEAMS),
            num_return_sequences=int(GENERATION_NUM_RETURN_SEQUENCES),
            early_stopping=bool(GENERATION_EARLY_STOPPING),
        )
    decoded = tokenizer.batch_decode(generated, skip_special_tokens=False)
    return [
        decoded[offset : offset + int(GENERATION_NUM_RETURN_SEQUENCES)]
        for offset in range(0, len(decoded), int(GENERATION_NUM_RETURN_SEQUENCES))
    ]


def _build_generation_group_and_row(example: dict[str, object], generated_texts: list[str], *, split_name: str) -> tuple[GenerationGroup, dict[str, object]]:
    candidate_inputs: list[MoleculeInput] = []
    parsed_selfies: list[str] = []
    cleaned_texts: list[str] = []
    generated_stage_counts: list[int] = []
    parse_failures = 0

    for sequence_index, raw_text in enumerate(generated_texts):
        cleaned_text = _clean_generated_sequence_text(raw_text, tokenizer)
        cleaned_texts.append(cleaned_text)
        molecules = parse_staged_target(cleaned_text, separator_token=" ")
        generated_stage_counts.append(len(molecules))
        if molecules:
            for molecule_index, selfies in enumerate(molecules):
                parsed_selfies.append(selfies)
                candidate_inputs.append(
                    MoleculeInput(
                        text=selfies,
                        representation="selfies",
                        molecule_id=f"sequence-{sequence_index:03d}-molecule-{molecule_index:03d}",
                    )
                )
        else:
            parse_failures += 1
            candidate_inputs.append(
                MoleculeInput(
                    text=cleaned_text,
                    representation="selfies",
                    molecule_id=f"sequence-{sequence_index:03d}-unparsed",
                )
            )

    targets = [str(selfies) for selfies in example["target_selfies_list"]]
    group = GenerationGroup(
        group_id=str(example["id"]),
        candidates=tuple(candidate_inputs),
        targets=tuple(MoleculeInput(text=selfies, representation="selfies") for selfies in targets),
    )
    row = {
        "id": str(example["id"]),
        "split": split_name,
        "description": str(example["description"]),
        "prompt": str(example["prompt"]),
        "target_selfies_list": targets,
        "raw_generated_texts": generated_texts,
        "cleaned_generated_texts": cleaned_texts,
        "generated_selfies_list": parsed_selfies,
        "generated_stage_counts": generated_stage_counts,
        "mean_generated_stage_count": sum(generated_stage_counts) / max(len(generated_stage_counts), 1),
        "mean_trajectory_length": sum(generated_stage_counts) / max(len(generated_stage_counts), 1),
        "num_candidate_inputs": len(candidate_inputs),
        "num_parsed_selfies": len(parsed_selfies),
        "num_parse_failures": parse_failures,
    }
    return group, row


def _compact_metrics(
    result,
    *,
    split_name: str,
    progress_examples: int | None = None,
    mean_generated_stage_count: float,
) -> dict[str, float | int]:
    prefix = f"eval/{split_name}"
    metrics = {
        f"{prefix}/accepted_unique_count": result.accepted_unique_count,
        f"{prefix}/n_circles": result.n_circles,
        f"{prefix}/internal_diversity": result.internal_diversity,
        f"{prefix}/novelty_fraction": result.novelty_fraction,
        f"{prefix}/novelty_count": result.novelty_count,
        f"{prefix}/valid_fraction": result.num_valid_candidates / max(result.num_candidates, 1),
        f"{prefix}/mean_max_dice_similarity": result.mean_max_dice_similarity,
        f"{prefix}/mean_generated_stage_count": float(mean_generated_stage_count),
        f"{prefix}/mean_trajectory_length": float(mean_generated_stage_count),
    }
    if progress_examples is not None:
        metrics[f"{prefix}/progress_examples"] = int(progress_examples)
    if split_name == "validation":
        metrics.update(
            {
                "eval/accepted_unique_count": result.accepted_unique_count,
                "eval/n_circles": result.n_circles,
                "eval/internal_diversity": result.internal_diversity,
                "eval/novelty_fraction": result.novelty_fraction,
                "eval/novelty_count": result.novelty_count,
                "eval/valid_fraction": result.num_valid_candidates / max(result.num_candidates, 1),
                "eval/mean_max_dice_similarity": result.mean_max_dice_similarity,
                "eval/mean_generated_stage_count": float(mean_generated_stage_count),
                "eval/mean_trajectory_length": float(mean_generated_stage_count),
            }
        )
    return metrics


def _selected_indices_snapshot(selected_indices: list[int], *, limit: int = 1000) -> dict[str, object]:
    truncated = len(selected_indices) > int(limit)
    return {
        "selected_indices": selected_indices[: int(limit)],
        "selected_indices_count": len(selected_indices),
        "selected_indices_truncated": truncated,
    }


def _append_jsonl(path: Path, row: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row) + "\n")


def _build_progress_payload(
    result,
    *,
    split_name: str,
    dataset_path: Path,
    dataset_size: int,
    num_selected_examples: int,
    evaluated_examples: int,
    selected_fraction: float | None,
    sample_with_replacement: bool,
    selected_indices: list[int],
    mean_generated_stage_count: float,
) -> dict[str, object]:
    valid_fraction = result.num_valid_candidates / max(result.num_candidates, 1)
    payload = {
        "split_name": split_name,
        "dataset_path": str(dataset_path),
        "dataset_size": dataset_size,
        "num_selected_examples": num_selected_examples,
        "evaluated_examples": evaluated_examples,
        "selected_fraction": selected_fraction,
        "selection_seed": int(EVAL_SEED),
        "sample_with_replacement": bool(sample_with_replacement),
        "eval_batch_size": int(EVAL_BATCH_SIZE),
        "generation_num_beams": int(GENERATION_NUM_BEAMS),
        "generation_num_return_sequences": int(GENERATION_NUM_RETURN_SEQUENCES),
        "accepted_unique_count": result.accepted_unique_count,
        "num_candidates": result.num_candidates,
        "num_valid_candidates": result.num_valid_candidates,
        "valid_fraction": valid_fraction,
        "internal_diversity": result.internal_diversity,
        "novelty_fraction": result.novelty_fraction,
        "mean_max_dice_similarity": result.mean_max_dice_similarity,
        "mean_generated_stage_count": float(mean_generated_stage_count),
        "mean_trajectory_length": float(mean_generated_stage_count),
    }
    payload.update(_selected_indices_snapshot(selected_indices))
    return payload


def evaluate_sft_dataset(
    dataset_path: Path,
    *,
    split_name: str,
    metrics_path: Path,
    generations_path: Path,
    progress_metrics_path: Path,
    max_examples: int | None = None,
    fraction: float | None = None,
    sample_with_replacement: bool = False,
    report_every_examples: int = 50,
) -> dict[str, object]:
    dataset = MultiMoleculeDataset.from_jsonl(dataset_path)
    examples, selected_indices = _select_examples(
        dataset,
        max_examples=max_examples,
        fraction=fraction,
        seed=EVAL_SEED,
        with_replacement=sample_with_replacement,
    )
    metric_config = EvaluationMetricConfig(
        acceptance_dice_threshold=float(ACCEPTANCE_DICE_THRESHOLD),
        n_circles_tanimoto_threshold=float(N_CIRCLES_TANIMOTO_THRESHOLD),
    )

    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    progress_metrics_path.parent.mkdir(parents=True, exist_ok=True)
    progress_metrics_path.write_text("", encoding="utf-8")

    groups: list[GenerationGroup] = []
    generation_rows: list[dict[str, object]] = []
    total_generated_sequences = 0
    total_generated_stage_count = 0
    report_every_examples = int(report_every_examples)
    should_report_progress = report_every_examples > 0

    example_batches = list(_batched(examples, EVAL_BATCH_SIZE))
    for example_batch in tqdm(example_batches, desc=f"Evaluating {split_name}", leave=False):
        generated_text_groups = _generate_text_batch(model, tokenizer, example_batch, device=device)
        for example, generated_texts in zip(example_batch, generated_text_groups):
            group, row = _build_generation_group_and_row(example, generated_texts, split_name=split_name)
            groups.append(group)
            generation_rows.append(row)
            total_generated_sequences += len(row["generated_stage_counts"])
            total_generated_stage_count += sum(int(value) for value in row["generated_stage_counts"])

            evaluated_examples = len(groups)
            if should_report_progress and evaluated_examples % report_every_examples == 0:
                partial_result = evaluate_generation_groups(groups, config=metric_config)
                partial_mean_generated_stage_count = total_generated_stage_count / max(total_generated_sequences, 1)
                progress_payload = _build_progress_payload(
                    partial_result,
                    split_name=split_name,
                    dataset_path=dataset_path,
                    dataset_size=len(dataset),
                    num_selected_examples=len(examples),
                    evaluated_examples=evaluated_examples,
                    selected_fraction=fraction,
                    sample_with_replacement=sample_with_replacement,
                    selected_indices=selected_indices,
                    mean_generated_stage_count=partial_mean_generated_stage_count,
                )
                _append_jsonl(progress_metrics_path, progress_payload)
                progress_metrics = _compact_metrics(
                    partial_result,
                    split_name=split_name,
                    progress_examples=evaluated_examples,
                    mean_generated_stage_count=partial_mean_generated_stage_count,
                )
                print(progress_metrics)
                if wandb.run is not None:
                    wandb.log(progress_metrics, step=evaluated_examples)

    result = evaluate_generation_groups(groups, config=metric_config)
    mean_generated_stage_count = total_generated_stage_count / max(total_generated_sequences, 1)
    payload = result.to_dict(include_assessments=False)
    payload.update(
        {
            "split_name": split_name,
            "dataset_path": str(dataset_path),
            "dataset_size": len(dataset),
            "num_examples": len(examples),
            "num_selected_examples": len(examples),
            "selection_fraction": fraction,
            "selection_seed": int(EVAL_SEED),
            "sample_with_replacement": bool(sample_with_replacement),
            "report_every_examples": report_every_examples,
            "eval_batch_size": int(EVAL_BATCH_SIZE),
            "generation_num_beams": int(GENERATION_NUM_BEAMS),
            "generation_num_return_sequences": int(GENERATION_NUM_RETURN_SEQUENCES),
            "generation_max_new_tokens": int(GENERATION_MAX_NEW_TOKENS),
            "mean_generated_stage_count": float(mean_generated_stage_count),
            "mean_trajectory_length": float(mean_generated_stage_count),
        }
    )
    payload.update(_selected_indices_snapshot(selected_indices))

    metrics_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    write_jsonl(generations_path, generation_rows)

    evaluation_diagnosis = _compact_metrics(
        result,
        split_name=split_name,
        progress_examples=len(groups),
        mean_generated_stage_count=mean_generated_stage_count,
    )
    print(evaluation_diagnosis)
    if wandb.run is not None:
        wandb.log(evaluation_diagnosis, step=len(groups))
    print({
        "split_name": split_name,
        "dataset_size": len(dataset),
        "num_selected_examples": len(examples),
        "metrics_path": str(metrics_path),
        "generations_path": str(generations_path),
        "progress_metrics_path": str(progress_metrics_path),
    })
    return payload


device = choose_device("auto")
checkpoint_for_eval = UPSTREAM_CHECKPOINT
if not checkpoint_for_eval.exists():
    raise RuntimeError(f"Missing multi-SFT checkpoint: {checkpoint_for_eval}")

tokenizer = AutoTokenizer.from_pretrained(checkpoint_for_eval, use_fast=True)
tokenizer.model_max_length = int(1e9)
model = T5ForConditionalGeneration.from_pretrained(checkpoint_for_eval)
model.to(device)
model.eval()

if wandb.run is None and WANDB_API_KEY:
    wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=RUN_NAME,
        job_type="sft_lpm24_evaluation",
    )

print({
    "checkpoint_for_eval": str(checkpoint_for_eval),
    "device": str(device),
    "tokenizer_vocab_size": len(tokenizer),
    "generation_num_beams": GENERATION_NUM_BEAMS,
    "generation_num_return_sequences": GENERATION_NUM_RETURN_SEQUENCES,
})


In [ ]:
validation_payload = evaluate_sft_dataset(
    VALIDATION_DATASET_PATH,
    split_name="validation",
    metrics_path=VALIDATION_METRICS_PATH,
    generations_path=VALIDATION_GENERATIONS_PATH,
    progress_metrics_path=VALIDATION_PROGRESS_METRICS_PATH,
    max_examples=VALIDATION_NUM_EXAMPLES,
    fraction=None,
    sample_with_replacement=True,
    report_every_examples=EVAL_REPORT_EVERY_EXAMPLES,
)
validation_payload


In [ ]:
processed_test_payload = evaluate_sft_dataset(
    PROCESSED_TEST_DATASET_PATH,
    split_name="processed_test",
    metrics_path=PROCESSED_TEST_METRICS_PATH,
    generations_path=PROCESSED_TEST_GENERATIONS_PATH,
    progress_metrics_path=PROCESSED_TEST_PROGRESS_METRICS_PATH,
    max_examples=PROCESSED_TEST_MAX_EXAMPLES,
    fraction=PROCESSED_TEST_FRACTION,
    sample_with_replacement=PROCESSED_TEST_SAMPLE_WITH_REPLACEMENT,
    report_every_examples=EVAL_REPORT_EVERY_EXAMPLES,
)
processed_test_payload


In [ ]:
import json

from src.checkpoint_bootstrap import archive_directory_to_zip

output_dir_zip = None
if OUTPUT_DIR.exists():
    output_dir_zip = archive_directory_to_zip(OUTPUT_DIR, OUTPUT_DIR_ZIP)

print({
    "output_dir": str(OUTPUT_DIR),
    "output_dir_exists": OUTPUT_DIR.exists(),
    "output_dir_zip": str(output_dir_zip) if output_dir_zip else str(OUTPUT_DIR_ZIP),
    "output_dir_zip_exists": output_dir_zip is not None and output_dir_zip.exists(),
    "validation_metrics_path": str(VALIDATION_METRICS_PATH),
    "validation_metrics_exists": VALIDATION_METRICS_PATH.exists(),
    "validation_generations_path": str(VALIDATION_GENERATIONS_PATH),
    "validation_generations_exists": VALIDATION_GENERATIONS_PATH.exists(),
    "validation_progress_metrics_path": str(VALIDATION_PROGRESS_METRICS_PATH),
    "validation_progress_metrics_exists": VALIDATION_PROGRESS_METRICS_PATH.exists(),
    "processed_test_metrics_path": str(PROCESSED_TEST_METRICS_PATH),
    "processed_test_metrics_exists": PROCESSED_TEST_METRICS_PATH.exists(),
    "processed_test_generations_path": str(PROCESSED_TEST_GENERATIONS_PATH),
    "processed_test_generations_exists": PROCESSED_TEST_GENERATIONS_PATH.exists(),
    "processed_test_progress_metrics_path": str(PROCESSED_TEST_PROGRESS_METRICS_PATH),
    "processed_test_progress_metrics_exists": PROCESSED_TEST_PROGRESS_METRICS_PATH.exists(),
    "upstream_checkpoint": str(UPSTREAM_CHECKPOINT),
    "upstream_checkpoint_exists": UPSTREAM_CHECKPOINT.exists(),
    "processed_dataset_checks": {name: path.exists() for name, path in PROCESSED_DATASET_CHECKS.items()},
    "grouped_split_checks": {name: path.exists() for name, path in GROUPED_SPLIT_CHECKS.items()},
})
